# Train and run PP-OCRv6 in Jupyter

Edit the image preprocessing, CTC loss and optimizer in the training cells. The inference section has a copyable TensorRT class that builds its own engine from the exported ONNX bundle. Choose Tiny, Small or Medium using MODEL_VARIANT below. The example trains the selected recognition model without the auxiliary training reader.

In [ ]:
from pathlib import Path
import os
import sys
import yaml

ROOT = next(path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (path / "configs/ppocr_price.yaml").is_file())
sys.path.insert(0, str(ROOT))
USE_ENERGY_METER = os.environ.get("PPOCR_ENERGY_METER", "0") == "1"
RUN_TRAINING = os.environ.get("PPOCR_RUN_TRAINING", "1") == "1"
RUN_TENSORRT = os.environ.get("PPOCR_RUN_TENSORRT", "0") == "1"
DATASET = ROOT / ("dataset/energy-meter" if USE_ENERGY_METER else "dummy_data/prices")
MODEL_VARIANT = os.environ.get("PPOCR_VARIANT", "medium")  # tiny, small, or medium
PRESETS = {"tiny": "ppocr_price_tiny", "small": "ppocr_price_small", "medium": "ppocr_price"}
cfg = yaml.safe_load((ROOT / "configs" / (PRESETS[MODEL_VARIANT] + ".yaml")).read_text())
dataset_name = "energy_meter" if USE_ENERGY_METER else "price"
WORK = Path(os.environ.get("PPOCR_OUTPUT", ROOT / "artifacts" / f"notebook_{dataset_name}_{MODEL_VARIANT}"))
cfg.update(dataset_root=DATASET, output_dir=WORK, pretrained_dir=ROOT / os.environ.get("PPOCR_PRETRAINED", cfg["pretrained_dir"]),
           epochs=1, batch_size=4, auxiliary=False)

## Inspect data and define the image transform

In [ ]:
"""Train-only selection then fresh full-fit PP-OCRv6-derived price recipe."""
from pathlib import Path
import contextlib, json, random, time
import numpy as np
import torch
from PIL import Image, ImageEnhance
from torch.utils.data import Dataset, DataLoader
from parseq_ocr.ppocr_models import VOCAB, create_model, restore_bn, preprocess, decode
from parseq_ocr.metrics import score_pairs, selection_score, EXACT_POLICY

from parseq_ocr.ppocr_data import canonical, scan_labeled, split_price_holdout


class Crops(Dataset):
    def __init__(self, rows, height, augment=False):
        self.rows = rows
        self.height = height
        self.augment = augment

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        with Image.open(r["path"]) as im:
            image = im.convert("RGB")
        if self.augment:
            if random.random() < 0.4:
                image = ImageEnhance.Brightness(image).enhance(random.uniform(0.8, 1.2))
            if random.random() < 0.4:
                image = ImageEnhance.Contrast(image).enhance(random.uniform(0.8, 1.2))
            if random.random() < 0.25:
                image = image.rotate(
                    random.uniform(-2, 2),
                    Image.Resampling.BILINEAR,
                    fillcolor=(127, 127, 127),
                )
        return torch.from_numpy(preprocess(image, self.height)), r["label"], r["image"]


def loader(rows, cfg, augment=False):
    return DataLoader(
        Crops(rows, cfg["height"], augment),
        batch_size=cfg.get("batch_size", 64),
        shuffle=augment,
        num_workers=cfg.get("workers", 0),
    )


def autocast(cfg):
    return (
        torch.autocast("cuda", dtype=torch.bfloat16)
        if cfg.get("device", "cuda") == "cuda"
        and cfg.get("precision", "bf16") == "bf16"
        else contextlib.nullcontext()
    )


@torch.inference_mode()
def evaluate_model(model, rows, cfg, output=None):
    model.eval()
    pred = []
    for x, labels, ids in loader(canonical(rows), cfg):
        with autocast(cfg):
            z = model(x.to(cfg.get("device", "cuda")))
        pred.extend(
            dict(image=i, gt=gt, prediction=t)
            for i, gt, t in zip(ids, labels, decode(z.argmax(-1).cpu().tolist()))
        )
    labels = [r["gt"] for r in pred]
    texts = [r["prediction"] for r in pred]
    metrics = {
        **score_pairs(labels, texts, "none"),
        "precision": cfg.get("precision", "bf16"),
    }
    if output:
        output = Path(output)
        output.mkdir(parents=True, exist_ok=True)
        (output / "predictions.jsonl").write_text(
            "".join(json.dumps(r) + "\n" for r in pred)
        )
        (output / "metrics.json").write_text(json.dumps(metrics, indent=2))
    return metrics



if not (DATASET / "val").is_dir():
    raise FileNotFoundError("Prepare the dataset or run the energy-meter downloader.")
if not torch.cuda.is_available():
    cfg.update(device="cpu", precision="fp32")
records = scan_labeled(DATASET / "train")
fit, holdout = split_price_holdout(records, cfg["holdout_fraction"], cfg["seed"])
validation = scan_labeled(DATASET / "val")
with Image.open(fit[0]["path"]) as image:
    display(image.copy())
pixels, texts, names = next(iter(loader(fit, cfg)))
print("Pixels:", tuple(pixels.shape), "Labels:", texts)

## Edit the CTC objective and training loop

The package supplies the pretrained model and data split helpers. The notebook owns optimization, epoch selection and checkpoint saving.

In [ ]:
def _fit(cfg, rows, held, epochs, folder):
    seed = cfg.get("seed", 20261114)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(cfg.get("cpu_threads", 4))
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    device = cfg.get("device", "cuda")
    model = create_model(cfg).to(device)
    if cfg.get("recover_bn", True):
        sample = random.Random(seed).sample(rows, min(cfg.get("calibration_samples", 128), len(rows)))
        calibration_cfg = {**cfg, "batch_size": cfg.get("calibration_batch_size", cfg.get("batch_size", 64))}
        restore_bn(model, (x for x, *_ in loader(sample, calibration_cfg)))
    parameters = sum(p.numel() for p in model.parameters())
    if cfg.get("auxiliary", True):
        from parseq_ocr.ppocr_auxiliary import AuxiliaryReader

        model = AuxiliaryReader(model, Path(cfg["pretrained_dir"]) / "parseq_tiny").to(
            device
        )
    opt = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.get("learning_rate", 1e-4),
        weight_decay=cfg.get("weight_decay", 3e-5),
    )
    ctc = torch.nn.CTCLoss(blank=0, zero_infinity=False)
    best = (-1.0, -1.0)
    selected = 0
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for x, labels, _ in loader(rows, cfg, True):
            target = torch.tensor(
                [VOCAB.index(c) for s in labels for c in s], device=device
            )
            lengths = torch.tensor([len(s) for s in labels], device=device)
            opt.zero_grad(set_to_none=True)
            with autocast(cfg):
                z = model(x.to(device))
            z = z.float().log_softmax(-1).transpose(0, 1)
            loss = ctc(
                z,
                target,
                torch.full((len(labels),), z.shape[0], device=device, dtype=torch.long),
                lengths,
            )
            if cfg.get("auxiliary", True):
                with autocast(cfg):
                    loss = loss + model.auxiliary_loss(labels)
            if not torch.isfinite(loss):
                raise RuntimeError("Nonfinite CTC/auxiliary loss")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 5.0, error_if_nonfinite=True
            )
            opt.step()
            losses.append(loss.item())
        row = {"epoch": epoch, "loss": sum(losses) / len(losses)}
        if held:
            row["holdout"] = evaluate_model(model, held, cfg)
            m = row["holdout"]
            score = selection_score(m)
        else:
            score = (epoch, epoch)
        if score > best:
            best = score
            selected = epoch
            base = model.base if cfg.get("auxiliary", True) else model
            torch.save(
                {
                    "state_dict": base.state_dict(),
                    "config": {
                        k: str(v) if isinstance(v, Path) else v for k, v in cfg.items()
                    },
                    "epoch": epoch,
                    "inference_parameters": parameters,
                    "training_parameters": sum(p.numel() for p in model.parameters()),
                    "recover_bn": cfg.get("recover_bn", True),
                    "selection_metric": "exact",
                    "selection_policy": EXACT_POLICY,
                    "full_fit": not bool(held),
                },
                folder / "model.pt",
            )
        history.append(row)
        (folder / "history.json").write_text(json.dumps(history, indent=2))
        print(json.dumps(row), flush=True)
    return selected


def train(config, fit_rows=None, held_rows=None):
    """Explicit manifests may reproduce a prior train-only fold; no val selection."""
    cfg = dict(config)
    if cfg.get("selection_metric", "exact") != "exact":
        raise ValueError("Only literal exact checkpoint selection is supported")
    if cfg.get("epochs", 0) < 1 or cfg.get("batch_size", 0) < 1:
        raise ValueError(
            "Positive epochs and batch_size required; existing checkpoints are never reused"
        )
    if fit_rows is None or held_rows is None:
        if fit_rows is not None or held_rows is not None:
            raise ValueError("Supply both fit_rows and held_rows")
        rows = scan_labeled(Path(cfg["dataset_root"]) / "train")
        fit_rows, held_rows = split_price_holdout(
            rows, cfg["holdout_fraction"], cfg.get("seed", 20261114)
        )
    fit, held = canonical(fit_rows), canonical(held_rows)
    if not fit or not held:
        raise ValueError("Nonempty fit and internal holdout required")
    if {str(Path(r["path"]).resolve()) for r in fit} & {
        str(Path(r["path"]).resolve()) for r in held
    }:
        raise ValueError("Fit and holdout paths overlap")
    for r in fit + held:
        s = r["label"]
        if not s or any(c not in VOCAB[1:] for c in s) or len(s) > 12:
            raise ValueError(
                f'Unsupported/overlength label:{r["image"]}; never truncated'
            )
    from parseq_ocr.data import audit_splits

    audit = audit_splits(fit, held)
    if audit["pixel_duplicates_across_splits"]:
        raise ValueError("Duplicate pixels across fit/holdout")
    output = Path(cfg["output_dir"])
    output.mkdir(parents=True, exist_ok=True)
    (output / "selection_split.json").write_text(
        json.dumps({"fit": fit, "holdout": held, "audit": audit}, indent=2)
    )
    selected = _fit(cfg, fit, held, cfg["epochs"], output / "selection")
    if cfg.get("refit", True):
        _fit(cfg, fit + held, [], selected, output / "full_fit")
    (output / "recipe.json").write_text(
        json.dumps(
            {
                "selected_epoch": selected,
                "fit_n": len(fit),
                "holdout_n": len(held),
                "full_fit_n": len(fit + held) if cfg.get("refit", True) else 0,
                "refit": cfg.get("refit", True),
                "selection_metric": "exact",
                    "selection_policy": EXACT_POLICY,
                "initialization": ("fresh pinned official recognition model for selection and fresh full refit; respective FIT-only BN calibration" if cfg.get("refit", True) else "fresh pinned official recognition model for FIT-only selection; no full refit; DEV never gradient/calibration"),
                "historical_note": "Preset uses exactinternalholdout selection; historicalwinnerselected15. Current dataset/precision/environment mayselectdifferentepoch.",
            },
            indent=2,
        )
    )
    return output / ("full_fit/model.pt" if cfg.get("refit", True) else "selection/model.pt")

## Train and review predictions

In [ ]:
from parseq_ocr.ppocr_models import prepare_pretrained
from parseq_ocr.ppocr_training import load_model

checkpoint = WORK / "full_fit/model.pt"
if RUN_TRAINING:
    if checkpoint.exists() or (WORK / "selection/model.pt").exists():
        raise FileExistsError("Choose a new PPOCR_OUTPUT or set RUN_TRAINING=False to reuse a checkpoint.")
    prepare_pretrained(cfg["pretrained_dir"], MODEL_VARIANT)
    checkpoint = train(cfg, fit, holdout)
if checkpoint.is_file():
    model, native_cfg = load_model(checkpoint, cfg["device"])
    print(evaluate_model(model, validation, native_cfg, WORK / "validation_native"))
    pred = (WORK / "validation_native/predictions.jsonl").read_text().splitlines()
    display([json.loads(line) for line in pred[:10]])

## Export ONNX

Set `RUN_TENSORRT = True` above to create the exported bundle and run the GPU example.

In [ ]:
from parseq_ocr.ppocr_export import export_onnx
DEPLOYMENT = WORK / "deployment"
if RUN_TENSORRT and not (DEPLOYMENT / "metadata.json").is_file():
    export_onnx(checkpoint, DEPLOYMENT)

## Copyable TensorRT runtime

Copy these cells in order to a Python file. They require the exported bundle plus TensorRT, CUDA Python, PyCUDA, Pillow, OpenCV and NumPy. The runtime cells can be copied without the training cells.

In [ ]:
from pathlib import Path
import hashlib
import json
def build_engine(deployment, max_batch=32, force=False):
    import tensorrt as trt

    folder = Path(deployment)
    metadata = json.loads((folder / "metadata.json").read_text())
    source = folder / "model.onnx"
    target = folder / "model.engine"
    if not 1 <= max_batch <= 32:
        raise ValueError("Validated batch range1..32")
    if target.exists() and not force:
        raise FileExistsError("Engine exists; force rebuild for currenttarget")
    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    net = builder.create_network(
        1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    )
    parser = trt.OnnxParser(net, logger)
    if not parser.parse(source.read_bytes()):
        raise RuntimeError(
            "\n".join(str(parser.get_error(i)) for i in range(parser.num_errors))
        )
    cfg = builder.create_builder_config()
    cfg.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
    cfg.set_flag(trt.BuilderFlag.FP16)
    if hasattr(trt.PreviewFeature, "RUNTIME_ACTIVATION_RESIZE_10_10"):
        cfg.set_preview_feature(
            trt.PreviewFeature.RUNTIME_ACTIVATION_RESIZE_10_10, True
        )
    profile = builder.create_optimization_profile()
    h, w = metadata["height"], metadata["width"]
    profile.set_shape(
        "pixels", (1, 3, h, w), (min(8, max_batch), 3, h, w), (max_batch, 3, h, w)
    )
    cfg.add_optimization_profile(profile)
    blob = builder.build_serialized_network(net, cfg)
    if blob is None:
        raise RuntimeError("TensorRT build failed")
    target.write_bytes(blob)
    (folder / "build.json").write_text(
        json.dumps(
            {
                "tensorrt": trt.__version__,
                "max_batch": max_batch,
                "precision": "FP16allowed",
                "activation_resize": True,
                "onnx_sha256": hashlib.sha256(source.read_bytes()).hexdigest(),
                "engine_sha256": hashlib.sha256(target.read_bytes()).hexdigest(),
            },
            indent=2,
        )
    )
    return target

In [ ]:
"""Torch-free PP price TensorRT with per-shape managed workspace and CUDAgraphs.
Apache-2.0 derivative; see _vendor/ppocr/NOTICE. Static128×320, batch1..32.
"""
import pathlib, json, math
import numpy as np, cv2
from PIL import Image


def checked(result):
    if result[0] != rt.cudaError_t.cudaSuccess:
        raise RuntimeError(str(result[0]))
    return result[1] if len(result) > 1 else None


class PPTRTManaged:
    def __init__(self, folder, max_batch=32):
        global cuda, trt, rt
        import pycuda.driver as cuda
        import tensorrt as trt
        from cuda.bindings import runtime as rt
        folder = pathlib.Path(folder)
        self.meta = json.loads((folder / "metadata.json").read_text())
        self.max_batch = max_batch
        self.height = self.meta["height"]
        self.width = self.meta["width"]
        cuda.init()
        self.cuda_context = cuda.Device(0).make_context()
        logger = trt.Logger(trt.Logger.ERROR)
        self.runtime = trt.Runtime(logger)
        self.engine = self.runtime.deserialize_cuda_engine(
            (folder / "model.engine").read_bytes()
        )
        assert self.engine is not None
        self.max_batch = min(
            max_batch, self.engine.get_tensor_profile_shape("pixels", 0)[2][0]
        )
        ctx = self.engine.create_execution_context(
            trt.ExecutionContextAllocationStrategy.USER_MANAGED
        )
        ctx.set_input_shape("pixels", (self.max_batch, 3, self.height, self.width))
        shape = tuple(ctx.get_tensor_shape("ids"))
        self.h_input = cuda.pagelocked_empty(
            (self.max_batch, 3, self.height, self.width), np.float32
        )
        self.h_output = cuda.pagelocked_empty(shape, np.int32)
        self.d_input = cuda.mem_alloc(self.h_input.nbytes)
        self.d_output = cuda.mem_alloc(self.h_output.nbytes)
        self.stream = cuda.Stream()
        self.graphs = {}
        self.cuda_context.pop()

    def capture(self, n):
        ctx = self.engine.create_execution_context(
            trt.ExecutionContextAllocationStrategy.USER_MANAGED
        )
        ctx.set_input_shape("pixels", (n, 3, self.height, self.width))
        required = ctx.update_device_memory_size_for_shapes()
        workspace = cuda.mem_alloc(max(required, 256))
        ctx.set_device_memory(int(workspace), max(required, 256))
        ctx.set_tensor_address("pixels", int(self.d_input))
        ctx.set_tensor_address("ids", int(self.d_output))
        cuda.memcpy_htod_async(self.d_input, self.h_input[:n], self.stream)
        ctx.execute_async_v3(self.stream.handle)
        cuda.memcpy_dtoh_async(self.h_output, self.d_output, self.stream)
        self.stream.synchronize()
        stream = rt.cudaStream_t(self.stream.handle)
        checked(
            rt.cudaStreamBeginCapture(
                stream, rt.cudaStreamCaptureMode.cudaStreamCaptureModeGlobal
            )
        )
        cuda.memcpy_htod_async(self.d_input, self.h_input[:n], self.stream)
        assert ctx.execute_async_v3(self.stream.handle)
        cuda.memcpy_dtoh_async(self.h_output, self.d_output, self.stream)
        graph = checked(rt.cudaStreamEndCapture(stream))
        ex = checked(rt.cudaGraphInstantiate(graph, 0))
        self.graphs[n] = (ctx, graph, ex, workspace, required)

    def infer(self, images):
        n = len(images)
        assert 0 < n <= self.max_batch
        for i, im in enumerate(images):
            if isinstance(im, Image.Image):
                im = np.asarray(im.convert("RGB"))[:, :, ::-1]
            else:
                im = im[:, :, ::-1]
            if self.meta["resize"] == "stretch":
                self.h_input[i] = (
                    cv2.resize(im, (self.width, self.height))
                    .transpose(2, 0, 1)
                    .astype(np.float32)
                    / 127.5
                    - 1
                )
            else:
                h, w = im.shape[:2]
                nw = min(self.width, math.ceil(w / h * self.height))
                self.h_input[i].fill(0)
                self.h_input[i, :, :, :nw] = (
                    cv2.resize(im, (nw, self.height))
                    .transpose(2, 0, 1)
                    .astype(np.float32)
                    / 127.5
                    - 1
                )
        self.cuda_context.push()
        try:
            if n not in self.graphs:
                self.capture(n)
            checked(
                rt.cudaGraphLaunch(
                    self.graphs[n][2], rt.cudaStream_t(self.stream.handle)
                )
            )
            self.stream.synchronize()
            texts = []
            for ids in self.h_output[:n].tolist():
                prev = -1
                out = []
                for x in ids:
                    if x and x != prev:
                        out.append(self.meta["vocabulary"][x])
                    prev = x
                texts.append("".join(out))
            return texts
        finally:
            self.cuda_context.pop()

    def close(self):
        if getattr(self, "cuda_context", None):
            self.cuda_context.push()
            workspaces = [v[3] for v in self.graphs.values()]
            for ctx, graph, ex, workspace, required in self.graphs.values():
                checked(rt.cudaGraphExecDestroy(ex))
                checked(rt.cudaGraphDestroy(graph))
            self.graphs.clear()
            ctx = None
            for workspace in workspaces:
                workspace.free()
            workspaces.clear()
            workspace = None
            self.engine = None
            self.runtime = None
            self.d_input.free()
            self.d_output.free()
            self.stream = None
            self.cuda_context.pop()
            self.cuda_context.detach()
            self.cuda_context = None


class TensorRTRecognizer(PPTRTManaged):
    def infer(self, images):
        images = list(images)
        out = []
        for at in range(0, len(images), self.max_batch):
            out.extend(super().infer(images[at : at + self.max_batch]))
        return out

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        self.close()

In [ ]:
class PPOCRTensorRT(TensorRTRecognizer):
    def __init__(self, deployment_dir, *, max_batch=32, force_rebuild=False):
        import tensorrt as trt

        folder = Path(deployment_dir)
        engine = folder / "model.engine"
        if force_rebuild or not engine.is_file():
            build_engine(folder, max_batch=max_batch, force=force_rebuild)
        else:
            receipt = json.loads((folder / "build.json").read_text())
            source = folder / "model.onnx"
            if (receipt["onnx_sha256"] != hashlib.sha256(source.read_bytes()).hexdigest()
                    or receipt["engine_sha256"] != hashlib.sha256(engine.read_bytes()).hexdigest()
                    or receipt["tensorrt"] != trt.__version__):
                raise ValueError("Cached engine does not match this ONNX/installation. Use force_rebuild=True.")
        super().__init__(folder, max_batch=max_batch)

    def predict(self, image):
        if isinstance(image, (str, Path)):
            with Image.open(image) as source:
                return self.infer([source.convert("RGB")])[0]
        return self.infer([image])[0]

## Build if needed and run TensorRT inference

In [ ]:
if RUN_TENSORRT:
    reader = PPOCRTensorRT(DEPLOYMENT)
    try:
        predictions = [reader.predict(row["path"]) for row in validation]
        print("Exact match:", sum(p == r["label"] for p, r in zip(predictions, validation, strict=True)),
              "/", len(validation))
        display(list(zip(predictions[:10], [r["label"] for r in validation[:10]])))
    finally:
        reader.close()